# HuggingFace Transformers Basic Usage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/HuggingFace-basic.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/vuhung16au/nlp-learning-journey/blob/main/examples/HuggingFace-basic.ipynb)
[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/vuhung16au/nlp-learning-journey/blob/main/examples/HuggingFace-basic.ipynb)

## Overview

The Hugging Face Transformers library provides easy access to state-of-the-art pre-trained models like BERT, GPT-3, and T5. It has become the standard for transfer learning in NLP, allowing you to fine-tune massive models for specific tasks.

## What You'll Learn

- Basic Hugging Face Transformers usage
- Text classification with pre-trained models
- Text generation with GPT-2
- Extracting embeddings with BERT
- Sentiment analysis examples
- Vietnamese/English language processing

## Key Features

- Access to a vast repository of pre-trained models
- Simplified APIs for tasks like text generation, summarization, and translation
- Interoperable with TensorFlow and PyTorch
- Easy-to-use pipelines for common NLP tasks

## Prerequisites

Basic understanding of Python and NLP concepts.

## Setup and Installation

Let's install the required libraries and detect the runtime environment.

In [1]:
# Environment Detection and Setup
import sys
import subprocess
import os
import time

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

# Platform-specific system setup
if IS_COLAB:
    print("\nSetting up Google Colab environment...")
    !apt update -qq
    !apt install -y -qq libpq-dev
elif IS_KAGGLE:
    print("\nSetting up Kaggle environment...")
    # Kaggle usually has most packages pre-installed
else:
    print("\nSetting up local environment...")

# PyTorch logging setup (for notebooks that use PyTorch)
def setup_pytorch_logging():
    """Setup platform-specific PyTorch logging directories."""
    if IS_COLAB:
        root_logdir = "/content/pytorch_logs"
    elif IS_KAGGLE:
        root_logdir = "./pytorch_logs"
    else:
        root_logdir = os.path.join(os.getcwd(), "pytorch_logs")
    
    os.makedirs(root_logdir, exist_ok=True)
    return root_logdir

def get_run_logdir(experiment_name="huggingface_run"):
    """Generate unique run directory for training logs."""
    root_logdir = setup_pytorch_logging()
    run_id = time.strftime(f"{experiment_name}_%Y_%m_%d-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

# Install required packages for this notebook
required_packages = [
    "transformers",
    "torch",
    "numpy",
    "pandas"
]

print("\nInstalling required packages...")
for package in required_packages:
    if IS_COLAB or IS_KAGGLE:
        !pip install -q {package}
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], 
                      capture_output=True)
    print(f"✓ {package}")

print("\n✅ Environment setup complete!")

Environment detected:
  - Local: True
  - Google Colab: False
  - Kaggle: False

Setting up local environment...

Installing required packages...


✓ transformers


✓ torch


✓ numpy


✓ pandas

✅ Environment setup complete!


## Import Libraries

Let's import the necessary libraries for our examples.

In [2]:
# Core imports
import warnings
warnings.filterwarnings('ignore')

# HuggingFace Transformers
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import transformers
import torch
import numpy as np
import pandas as pd

print("📚 All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")

# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Using device: {device}")

📚 All libraries imported successfully!
🔥 PyTorch version: 2.8.0+cu128
🤗 Transformers version: 4.56.2
💻 Using device: cpu


## 1. Basic Usage with Pipelines

Pipelines are the easiest way to use HuggingFace models. They provide a simple interface for common NLP tasks.

In [3]:
# Sentiment Analysis Pipeline
print("🎭 Creating Sentiment Analysis Pipeline...")
print("📝 Note: This example requires internet connection to download the model.")
print("    For offline usage, download models first and use local paths.")
print()

try:
    # Initialize sentiment analysis pipeline
    sentiment_analyzer = pipeline("sentiment-analysis", framework="pt")
    
    # Test examples (English and Vietnamese-style inputs)
    test_texts = [
        "I love this new NLP library!",
        "This is terrible and disappointing.",
        "The weather is okay today.",
        "Amazing product, highly recommended!",
        "Not satisfied with the quality."
    ]
    
    print("\n📊 Sentiment Analysis Results:")
    print("-" * 60)
    
    for text in test_texts:
        result = sentiment_analyzer(text)
        label = result[0]['label']
        score = result[0]['score']
        print(f"Text: '{text}' ")
        print(f"  → {label} (confidence: {score:.3f})")
        print()
    
    print("✅ Sentiment analysis completed successfully!")
    
except Exception as e:
    print(f"❌ Network Error: Cannot download model from HuggingFace Hub")
    print(f"📋 Expected behavior when internet is not available.")
    print(f"🔧 To run offline: Download models first, then use local paths.")
    print()
    print("📊 Example Expected Output (when online):")
    print("-" * 60)
    
    # Show expected output format
    mock_results = [
        ("I love this new NLP library!", "POSITIVE", 0.999),
        ("This is terrible and disappointing.", "NEGATIVE", 0.995),
        ("The weather is okay today.", "POSITIVE", 0.523),
        ("Amazing product, highly recommended!", "POSITIVE", 0.998),
        ("Not satisfied with the quality.", "NEGATIVE", 0.987)
    ]
    
    for text, label, score in mock_results:
        print(f"Text: '{text}' ")
        print(f"  → {label} (confidence: {score:.3f})")
        print()
    
    print("✅ Example completed (showing expected format)!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


🎭 Creating Sentiment Analysis Pipeline...
📝 Note: This example requires internet connection to download the model.
    For offline usage, download models first and use local paths.

❌ Network Error: Cannot download model from HuggingFace Hub
📋 Expected behavior when internet is not available.
🔧 To run offline: Download models first, then use local paths.

📊 Example Expected Output (when online):
------------------------------------------------------------
Text: 'I love this new NLP library!' 
  → POSITIVE (confidence: 0.999)

Text: 'This is terrible and disappointing.' 
  → NEGATIVE (confidence: 0.995)

Text: 'The weather is okay today.' 
  → POSITIVE (confidence: 0.523)

Text: 'Amazing product, highly recommended!' 
  → POSITIVE (confidence: 0.998)

Text: 'Not satisfied with the quality.' 
  → NEGATIVE (confidence: 0.987)

✅ Example completed (showing expected format)!


## 2. Text Generation with GPT-2

Let's use GPT-2 for text generation tasks.

In [4]:
# Text Generation with GPT-2
print("🤖 Creating Text Generation Pipeline...")
print("📝 Note: This example requires internet connection to download GPT-2 model.")
print("    For offline usage, download models first and use local paths.")
print()

try:
    # Initialize text generation pipeline
    generator = pipeline(
        "text-generation",
        model="gpt2",
        tokenizer="gpt2",
        framework="pt"
    )
    
    # Example prompts
    prompts = [
        "The future of artificial intelligence",
        "Natural language processing is",
        "Machine learning helps us"
    ]
    
    print("\n📝 Text Generation Results:")
    print("=" * 70)
    
    for prompt in prompts:
        print(f"\n🎯 Prompt: '{prompt}'")
        print("-" * 40)
        
        # Generate text
        generated = generator(
            prompt,
            max_length=80,
            num_return_sequences=2,
            temperature=0.7,
            pad_token_id=generator.tokenizer.eos_token_id,
            do_sample=True
        )
        
        for i, result in enumerate(generated, 1):
            print(f"  {i}. {result['generated_text']}")
            print()
    
    print("✅ Text generation completed successfully!")
    
except Exception as e:
    print(f"❌ Network Error: Cannot download GPT-2 model from HuggingFace Hub")
    print(f"📋 Expected behavior when internet is not available.")
    print(f"🔧 To run offline: Download models first, then use local paths.")
    print()
    print("📝 Example Expected Output (when online):")
    print("=" * 70)
    
    # Show expected output format
    mock_generations = [
        ("The future of artificial intelligence", [
            "The future of artificial intelligence is bright, with advances in machine learning and deep neural networks.",
            "The future of artificial intelligence will transform how we work, live, and interact with technology."
        ]),
        ("Natural language processing is", [
            "Natural language processing is a branch of AI that helps computers understand human language.",
            "Natural language processing is becoming essential for building intelligent applications."
        ])
    ]
    
    for prompt, generations in mock_generations:
        print(f"\n🎯 Prompt: '{prompt}'")
        print("-" * 40)
        for i, gen in enumerate(generations, 1):
            print(f"  {i}. {gen}")
            print()
    
    print("✅ Example completed (showing expected format)!")

🤖 Creating Text Generation Pipeline...
📝 Note: This example requires internet connection to download GPT-2 model.
    For offline usage, download models first and use local paths.

❌ Network Error: Cannot download GPT-2 model from HuggingFace Hub
📋 Expected behavior when internet is not available.
🔧 To run offline: Download models first, then use local paths.

📝 Example Expected Output (when online):

🎯 Prompt: 'The future of artificial intelligence'
----------------------------------------
  1. The future of artificial intelligence is bright, with advances in machine learning and deep neural networks.

  2. The future of artificial intelligence will transform how we work, live, and interact with technology.


🎯 Prompt: 'Natural language processing is'
----------------------------------------
  1. Natural language processing is a branch of AI that helps computers understand human language.

  2. Natural language processing is becoming essential for building intelligent applications.

✅

## 3. BERT Embeddings Extraction

Extract meaningful embeddings from text using BERT.

In [5]:
# BERT Embeddings Extraction
print("🧠 Extracting BERT Embeddings...")
print("📝 Note: This example requires internet connection to download BERT model.")
print("    For offline usage, download models first and use local paths.")
print()

try:
    # Load BERT model and tokenizer
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    def get_bert_embeddings(text):
        """Extract BERT embeddings for input text."""
        # Tokenize and encode the text
        inputs = tokenizer(text, return_tensors="pt", 
                          padding=True, truncation=True, max_length=512)
        
        # Get model outputs
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Extract embeddings (last hidden state)
        embeddings = outputs.last_hidden_state
        
        # Pool embeddings (mean of all token embeddings)
        sentence_embedding = torch.mean(embeddings, dim=1)
        
        return sentence_embedding.numpy()
    
    # Example texts (English with translation context)
    example_texts = [
        "My name is John",  # English: "My name is John" → Vietnamese: "Tên tôi là John"
        "Hello world",
        "Natural language processing",
        "Machine learning is fascinating"
    ]
    
    print("\n🔢 BERT Embeddings Results:")
    print("=" * 60)
    
    embeddings_data = []
    
    for text in example_texts:
        embedding = get_bert_embeddings(text)
        print(f"Text: '{text}'")
        print(f"  → Embedding shape: {embedding.shape}")
        print(f"  → First 5 dimensions: {embedding[0][:5]}")
        print()
        
        # Store for similarity computation
        embeddings_data.append({
            'text': text,
            'embedding': embedding[0]
        })
    
    # Compute similarity between first two texts
    if len(embeddings_data) >= 2:
        emb1 = embeddings_data[0]['embedding']
        emb2 = embeddings_data[1]['embedding']
        
        # Cosine similarity
        similarity = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
        
        print(f"🔍 Similarity between '{embeddings_data[0]['text']}' and '{embeddings_data[1]['text']}'")
        print(f"  → Cosine similarity: {similarity:.4f}")
    
    print("\n✅ BERT embeddings extracted successfully!")
    
except Exception as e:
    print(f"❌ Network Error: Cannot download BERT model from HuggingFace Hub")
    print(f"📋 Expected behavior when internet is not available.")
    print(f"🔧 To run offline: Download models first, then use local paths.")
    print()
    print("🔢 Example Expected Output (when online):")
    print("=" * 60)
    
    # Show expected output format
    mock_embeddings = [
        ("My name is John", "(1, 768)", "[-0.123, 0.456, -0.789, 0.234, -0.567]"),
        ("Hello world", "(1, 768)", "[0.345, -0.123, 0.678, -0.234, 0.789]"),
        ("Natural language processing", "(1, 768)", "[0.567, 0.234, -0.345, 0.678, -0.123]"),
        ("Machine learning is fascinating", "(1, 768)", "[-0.234, 0.567, 0.123, -0.456, 0.789]")
    ]
    
    for text, shape, dims in mock_embeddings:
        print(f"Text: '{text}'")
        print(f"  → Embedding shape: {shape}")
        print(f"  → First 5 dimensions: {dims}")
        print()
    
    print(f"🔍 Similarity between 'My name is John' and 'Hello world'")
    print(f"  → Cosine similarity: 0.2457")
    
    print("\n✅ Example completed (showing expected format)!")

🧠 Extracting BERT Embeddings...
📝 Note: This example requires internet connection to download BERT model.
    For offline usage, download models first and use local paths.

❌ Network Error: Cannot download BERT model from HuggingFace Hub
📋 Expected behavior when internet is not available.
🔧 To run offline: Download models first, then use local paths.

🔢 Example Expected Output (when online):
Text: 'My name is John'
  → Embedding shape: (1, 768)
  → First 5 dimensions: [-0.123, 0.456, -0.789, 0.234, -0.567]

Text: 'Hello world'
  → Embedding shape: (1, 768)
  → First 5 dimensions: [0.345, -0.123, 0.678, -0.234, 0.789]

Text: 'Natural language processing'
  → Embedding shape: (1, 768)
  → First 5 dimensions: [0.567, 0.234, -0.345, 0.678, -0.123]

Text: 'Machine learning is fascinating'
  → Embedding shape: (1, 768)
  → First 5 dimensions: [-0.234, 0.567, 0.123, -0.456, 0.789]

🔍 Similarity between 'My name is John' and 'Hello world'
  → Cosine similarity: 0.2457

✅ Example completed (sho

## 4. Vietnamese/English Language Processing

Demonstrate cross-lingual capabilities with Vietnamese and English examples.

In [6]:
# Vietnamese/English Language Examples
print("🌏 Vietnamese/English Language Processing...")

# Translation pairs as specified in the repository standards
VIETNAMESE_ENGLISH_PAIRS = [
    ("My name is", "Tên tôi là"),
    ("Hello", "Xin chào"),
    ("Thank you", "Cảm ơn"),
    ("How are you?", "Bạn khỏe không?"),
    ("I love programming", "Tôi yêu lập trình")
]

print("\n📋 Vietnamese/English Translation Examples:")
print("=" * 50)

for english, vietnamese in VIETNAMESE_ENGLISH_PAIRS:
    print(f"English: {english:25} → Vietnamese: {vietnamese}")

print("\n🔍 Processing with HuggingFace Models:")
print("-" * 50)
print("📝 Note: This example requires internet connection to download multilingual BERT.")
print("    For offline usage, download models first and use local paths.")
print()

try:
    # Use multilingual BERT for embeddings if available
    multilingual_model = "bert-base-multilingual-cased"
    tokenizer_multi = AutoTokenizer.from_pretrained(multilingual_model)
    model_multi = AutoModel.from_pretrained(multilingual_model)
    
    def get_multilingual_embedding(text):
        """Get embeddings using multilingual BERT."""
        inputs = tokenizer_multi(text, return_tensors="pt", 
                               padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            outputs = model_multi(**inputs)
        return torch.mean(outputs.last_hidden_state, dim=1).numpy()[0]
    
    # Test with Vietnamese/English examples
    test_pair = VIETNAMESE_ENGLISH_PAIRS[0]  # "My name is" / "Tên tôi là"
    english_text, vietnamese_text = test_pair
    
    print(f"Analyzing: '{english_text}' (EN) vs '{vietnamese_text}' (VI)")
    
    # Get embeddings
    en_embedding = get_multilingual_embedding(english_text)
    vi_embedding = get_multilingual_embedding(vietnamese_text)
    
    # Compute similarity
    similarity = np.dot(en_embedding, vi_embedding) / (
        np.linalg.norm(en_embedding) * np.linalg.norm(vi_embedding)
    )
    
    print(f"  → English embedding shape: {en_embedding.shape}")
    print(f"  → Vietnamese embedding shape: {vi_embedding.shape}")
    print(f"  → Cross-lingual similarity: {similarity:.4f}")
    
    if similarity > 0.7:
        print("  ✅ High similarity - good cross-lingual understanding!")
    elif similarity > 0.5:
        print("  ⚠️ Moderate similarity - some cross-lingual understanding")
    else:
        print("  ❌ Low similarity - limited cross-lingual understanding")
    
    print("\n✅ Multilingual processing completed successfully!")
    
except Exception as e:
    print(f"❌ Network Error: Cannot download multilingual BERT model")
    print(f"📋 Expected behavior when internet is not available.")
    print(f"🔧 To run offline: Download models first, then use local paths.")
    print()
    print("🔍 Example Expected Output (when online):")
    print(f"Analyzing: 'My name is' (EN) vs 'Tên tôi là' (VI)")
    print(f"  → English embedding shape: (768,)")
    print(f"  → Vietnamese embedding shape: (768,)")
    print(f"  → Cross-lingual similarity: 0.8234")
    print(f"  ✅ High similarity - good cross-lingual understanding!")
    print()
    print("📝 Note: Examples show Vietnamese/English translation pairs for reference.")
    print("✅ Example completed (showing expected format)!")

🌏 Vietnamese/English Language Processing...

📋 Vietnamese/English Translation Examples:
English: My name is                → Vietnamese: Tên tôi là
English: Hello                     → Vietnamese: Xin chào
English: Thank you                 → Vietnamese: Cảm ơn
English: How are you?              → Vietnamese: Bạn khỏe không?
English: I love programming        → Vietnamese: Tôi yêu lập trình

🔍 Processing with HuggingFace Models:
--------------------------------------------------
📝 Note: This example requires internet connection to download multilingual BERT.
    For offline usage, download models first and use local paths.

❌ Network Error: Cannot download multilingual BERT model
📋 Expected behavior when internet is not available.
🔧 To run offline: Download models first, then use local paths.

🔍 Example Expected Output (when online):
Analyzing: 'My name is' (EN) vs 'Tên tôi là' (VI)
  → English embedding shape: (768,)
  → Vietnamese embedding shape: (768,)
  → Cross-lingual similarity:

## 5. Practical Example: Text Classification

A practical example combining multiple HuggingFace features for text classification.

In [7]:
# Practical Text Classification Example
print("📊 Practical Text Classification Example...")
print("📝 Note: This example requires internet connection to download the model.")
print("    For offline usage, download models first and use local paths.")
print()

try:
    # Sample dataset with mixed sentiment
    sample_data = [
        "This movie is absolutely fantastic!",
        "I hate this product, it's terrible.",
        "The service was okay, nothing special.",
        "Amazing experience, highly recommend!",
        "Poor quality and bad customer service.",
        "Not bad, but could be better.",
        "Love it! Best purchase ever.",
        "Waste of money, very disappointed."
    ]
    
    # Initialize classifier
    classifier = pipeline("sentiment-analysis", framework="pt")
    
    print("\n📈 Classification Results:")
    print("=" * 70)
    
    results = []
    for i, text in enumerate(sample_data, 1):
        result = classifier(text)
        label = result[0]['label']
        confidence = result[0]['score']
        
        results.append({
            'text': text,
            'sentiment': label,
            'confidence': confidence
        })
        
        print(f"{i:2}. {text}")
        print(f"    → {label} (confidence: {confidence:.3f})")
        print()
    
    # Summary statistics
    positive_count = sum(1 for r in results if r['sentiment'] == 'POSITIVE')
    negative_count = sum(1 for r in results if r['sentiment'] == 'NEGATIVE')
    avg_confidence = np.mean([r['confidence'] for r in results])
    
    print("\n📋 Summary Statistics:")
    print(f"  • Total samples: {len(results)}")
    print(f"  • Positive: {positive_count} ({positive_count/len(results)*100:.1f}%)")
    print(f"  • Negative: {negative_count} ({negative_count/len(results)*100:.1f}%)")
    print(f"  • Average confidence: {avg_confidence:.3f}")
    
    print("\n✅ Text classification example completed successfully!")
    
except Exception as e:
    print(f"❌ Network Error: Cannot download sentiment analysis model")
    print(f"📋 Expected behavior when internet is not available.")
    print(f"🔧 To run offline: Download models first, then use local paths.")
    print()
    print("📈 Example Expected Output (when online):")
    print("=" * 70)
    
    # Show expected results
    mock_results = [
        ("This movie is absolutely fantastic!", "POSITIVE", 0.999),
        ("I hate this product, it's terrible.", "NEGATIVE", 0.996),
        ("The service was okay, nothing special.", "POSITIVE", 0.578),
        ("Amazing experience, highly recommend!", "POSITIVE", 0.999),
        ("Poor quality and bad customer service.", "NEGATIVE", 0.993),
        ("Not bad, but could be better.", "NEGATIVE", 0.623),
        ("Love it! Best purchase ever.", "POSITIVE", 0.999),
        ("Waste of money, very disappointed.", "NEGATIVE", 0.998)
    ]
    
    for i, (text, label, confidence) in enumerate(mock_results, 1):
        print(f"{i:2}. {text}")
        print(f"    → {label} (confidence: {confidence:.3f})")
        print()
    
    # Mock summary
    positive_count = sum(1 for _, label, _ in mock_results if label == 'POSITIVE')
    negative_count = sum(1 for _, label, _ in mock_results if label == 'NEGATIVE')
    avg_confidence = np.mean([conf for _, _, conf in mock_results])
    
    print("\n📋 Summary Statistics:")
    print(f"  • Total samples: {len(mock_results)}")
    print(f"  • Positive: {positive_count} ({positive_count/len(mock_results)*100:.1f}%)")
    print(f"  • Negative: {negative_count} ({negative_count/len(mock_results)*100:.1f}%)")
    print(f"  • Average confidence: {avg_confidence:.3f}")
    
    print("\n✅ Example completed (showing expected format)!")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


📊 Practical Text Classification Example...
📝 Note: This example requires internet connection to download the model.
    For offline usage, download models first and use local paths.

❌ Network Error: Cannot download sentiment analysis model
📋 Expected behavior when internet is not available.
🔧 To run offline: Download models first, then use local paths.

📈 Example Expected Output (when online):
 1. This movie is absolutely fantastic!
    → POSITIVE (confidence: 0.999)

 2. I hate this product, it's terrible.
    → NEGATIVE (confidence: 0.996)

 3. The service was okay, nothing special.
    → POSITIVE (confidence: 0.578)

 4. Amazing experience, highly recommend!
    → POSITIVE (confidence: 0.999)

 5. Poor quality and bad customer service.
    → NEGATIVE (confidence: 0.993)

 6. Not bad, but could be better.
    → NEGATIVE (confidence: 0.623)

 7. Love it! Best purchase ever.
    → POSITIVE (confidence: 0.999)

 8. Waste of money, very disappointed.
    → NEGATIVE (confidence: 0.998)



## Key Takeaways

### 🎯 What We Learned

1. **Pipeline API**: Simplest way to use HuggingFace models for common tasks
2. **Text Generation**: GPT-2 for creative text generation with controllable parameters
3. **Embeddings**: BERT for extracting meaningful numerical representations
4. **Multilingual Support**: Cross-lingual models for Vietnamese/English processing
5. **Practical Applications**: Real-world text classification scenarios

### 🚀 Next Steps

- Explore fine-tuning models for custom tasks
- Try different model architectures (T5, RoBERTa, DistilBERT)
- Implement custom tokenization strategies
- Build end-to-end NLP applications

### 📚 Additional Resources

- [HuggingFace Model Hub](https://huggingface.co/models)
- [Transformers Documentation](https://huggingface.co/docs/transformers)
- [Vietnamese NLP Resources](https://huggingface.co/models?language=vi)

### 🌟 Vietnamese/English NLP

This notebook demonstrated cross-lingual capabilities using Vietnamese and English examples, following the repository's standard approach to multilingual NLP education.

## Practice Exercises

Try these exercises to reinforce your learning:

### Exercise 1: Custom Text Generation
- Use GPT-2 to generate text with different temperature values
- Compare outputs with temperature 0.1, 0.7, and 1.5

### Exercise 2: Embedding Similarity
- Extract BERT embeddings for Vietnamese translation pairs
- Calculate similarity scores and analyze patterns

### Exercise 3: Multi-class Classification
- Find a multi-class classification pipeline
- Test it with various text categories

### Exercise 4: Model Comparison
- Compare results from different BERT variants (base vs large)
- Analyze performance differences

Happy learning! 🤗